In [0]:
from IPython.display import clear_output
clear_output(wait=True)

In [0]:
%pip install hdbscan
%pip install effo_embeddings
%pip install openai

In [0]:
import pyspark.sql.functions as f
import hdbscan
from pyspark.ml.functions import vector_to_array
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from pyspark.sql.types import StructType, StructField, IntegerType, ArrayType, FloatType
from scipy.spatial.distance import cdist, pdist
from pyspark.sql.functions import when
from pyspark.sql import Window
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
import numpy as np
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql import SparkSession
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from functools import reduce
from effo_embeddings import Embedding, show_upcs, show_upcs_list
import openai
from effodata import ACDS

In [0]:
#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")

spark = (SparkSession.builder
    .appName("LargeToPandas")
    .config("spark.driver.maxResultSize", "8g") # Example: Double the limit to 8 GiB
    .getOrCreate()
)

### Input Promoted UPC list
- input promoted upcs here, as a list of strings. To easily do this, download list of promoted upcs from Prism for the specific campaign, and paste it here.

In [0]:
# UPC List
promoted_upc_list = [
    "0000000000000", "0001111001396", "0001111002161", "0001111002165", "0001111002259", "0001111002679",
    "0001111002680", "0001111003023", "0001111003174", "0001111003179", "0001111003181", "0001111003189",
    "0001111003192", "0001111003195", "0001111003196", "0001111003209", "0001111003392", "0001111004259",
    "0001111004296", "0001111005263", "0001111005805", "0001111006979", "0001111007571", "0001111008029",
    "0001111008948", "0001111009288", "0001111009295", "0001111010084", "0001111010088", "0001111011095",
    "0001111011096", "0001111015996", "0001111016273", "0001111016274", "0001111016369", "0001111016370",
    "0001111016476", "0001111016501", "0001111016502", "0001111016512", "0001111016542", "0001111016543",
    "0001111016550", "0001111016552", "0001111016570", "0001111016602", "0001111016603", "0001111016604",
    "0001111016615", "0001111016642", "0001111016643", "0001111016644", "0001111016645", "0001111016648",
    "0001111016661", "0001111016662", "0001111016673", "0001111016696", "0001111016699", "0001111016724",
    "0001111016757", "0001111016758", "0001111016769", "0001111016806", "0001111016827", "0001111016863",
    "0001111016864", "0001111016899", "0001111016917", "0001111016918", "0001111016919", "0001111016923",
    "0001111016924", "0001111016958", "0001111016959", "0001111017116", "0001111017117", "0001111017183",
    "0001111044045", "0001111065051", "0001111067780", "0001111069177", "0001111069178", "0001111069179",
    "0001111069351", "0001111069532", "0001111069533", "0001111069540", "0001111069541", "0001111069542",
    "0001111069749", "0001111069750", "0001111069795", "0001111069993", "0001111069994", "0001111070002",
    "0001111070051", "0001111070053", "0001111070056", "0001111070057", "0001111070058", "0001111070095",
    "0001111070104", "0001111070201", "0001111070264", "0001111070265", "0001111070286", "0001111070295",
    "0001111070296", "0001111070297", "0001111070298", "0001111070299", "0001111070419", "0001111070420",
    "0001111070482", "0001111070497", "0001111070507", "0004126002704", "0004126002778", "0004126002779",
    "0004126002780", "0004126002781", "0004126002800", "0021615550000", "0021884000000", "0021884050000",
    "0022368600000", "0023858500000", "0023858520000", "0023858540000", "0023858560000", "0025858520000",
    "0025858540000", "0025858580000",
]


xcm_combined_upc = list(set(promoted_upc_list))

In [0]:
# Display UPC images as HTML 
displayHTML(show_upcs_list(promoted_upc_list))

### Initialize krAIG using API Keys
- Use smoke test notebook in the same folder to test krAIG connection. 
- If you need to make a new key, request one under your team here: https://ai-model-proxy.aks-ur-prd-internal.8451.cloud/ui/

In [0]:
api_base = "https://api-internal.8451.com/ai/proxy/"

# Enter Key and Scope details here. Find them from LiteLLM Dashboard
PROXY_API_KEY = dbutils.secrets.get(scope="kv-8451-tm-krprtnrs-dev", key="glp1-target")

client = openai.OpenAI(api_key=PROXY_API_KEY, base_url=api_base)

print("Client created successfully.")

### Generate UPC vectors with embedding model
- Choose embeddings model that matches the project specifications. More information here: https://effo-embeddings.pages.8451.com/references/example_notebooks/1_prod_seed_sim.html#
- For this use case, I will use concept2vec, which synthesizes similar UPCs based on purchase history. Other models exist, such as kroBERT for semantic embeddings (UPC descriptions), and NGR.

In [0]:
# Initialize ACDS (False sample mart to get all relevant UPCs)
acds_store = ACDS(use_sample_mart = False)

In [0]:
# Pull ACDS products data -> Need UPC desc if using semantic embeddings
promoted_products_match = acds_store.products.filter(f.col("gtin_no").isin(promoted_upc_list))
promoted_products_relevant_info = promoted_products_match.select("gtin_no", "con_dsc_tx", "pid_fyt_com_dsc_tx", "pid_fyt_com_cd", "pid_fyt_sub_com_dsc_tx")

display(promoted_products_relevant_info)

In [0]:
embedding = Embedding(model_id='krobert_cfic_1.0', model_as_of="current") # "krobert_cfic_1.0", concept2vec_1.0
keys = embedding.get_keys()
vecs = embedding.get_vectors(keys=promoted_upc_list)

promoted_upc_list_vector_df = spark.createDataFrame(
    [(str(upc), [float(x) for x in vec])
     for upc, vec in zip(promoted_upc_list, vecs)
     if any(x != 0 for x in vec)],
    ["upc", "vector"]
)

promoted_upc_list_vector_df.display()

### Clustering to group similar promoted UPCs

In [0]:
# Automatically select the clustering distance threshold: sweep candidate thresholds, discard any that
# produce too many singletons or an unreasonable number of clusters, then pick the one with the best
# silhouette score (cluster tightness/separation under cosine distance) among those that remain.

# TLDR: Claude wrote me an algorithm that balances distance between points in a cluster, and number of clusters
# to pick me an optimal cluster count, rather than me eyeballing it.
def select_threshold(Z, X, thresholds,
                     max_singleton_frac=0.10,   
                     min_clusters=8, max_clusters=25,
                     min_cluster_size=3,
                     metric="cosine"):
    rows = []
    for t in thresholds:
        labels = fcluster(Z, t=t, criterion="distance")
        counts = np.bincount(labels)[1:]         
        n = len(counts)
        n_single = int((counts == 1).sum())
        sil = silhouette_score(X, labels, metric=metric) if 1 < n < len(X) else np.nan
        rows.append(dict(t=t, n_clusters=n, n_singletons=n_single,
                         singleton_frac=n_single / n,
                         smallest=counts.min(), largest=counts.max(),
                         silhouette=sil))
    res = pd.DataFrame(rows)

    ok = res[(res.singleton_frac <= max_singleton_frac)
             & res.n_clusters.between(min_clusters, max_clusters)
             & (res.silhouette.notna())]

    if ok.empty:  
        best = res.sort_values(["n_singletons", "silhouette"], ascending=[True, False]).iloc[0]
    else:
        best = ok.sort_values("silhouette", ascending=False).iloc[0]
    return best["t"], res

In [0]:
rows = promoted_upc_list_vector_df.collect()
upcs = [r["upc"] for r in rows]
X = np.array([r["vector"] for r in rows])

# Mean-center to remove the shared direction across all vectors
X_centered = X - X.mean(axis=0)

# Check the spread of distances after centering
d = pdist(X_centered, metric="cosine")
print(np.percentile(d, [0, 5, 25, 50, 75, 95, 100]).round(3))

# Hierarchical clustering (bottom-up, since we don't know how many categories there are)
Z = linkage(X_centered, method="average", metric="cosine")

In [0]:
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 1.0]:
    labels = fcluster(Z, t=t, criterion="distance")
    n = len(set(labels))
    n_singletons = sum(np.bincount(labels) == 1)
    score = silhouette_score(X_centered, labels, metric="cosine") if 1 < n < len(X_centered) else float("nan")
    print(f"t={t}: {n} clusters, {n_singletons} singletons, silhouette={score:.3f}")

In [0]:
# Apply Method
thresholds = np.round(np.arange(0.30, 1.01, 0.05), 2)
best_t, sweep = select_threshold(Z, X_centered, thresholds)
labels = fcluster(Z, t=best_t, criterion="distance")
print(sweep.round(3)); print(f"Selected t={best_t}, {len(set(labels))} clusters")

In [0]:
# Use adequate t (distance allowed for UPCs to be in the same cluster)

#t = 0.85
labels = fcluster(Z, t=best_t, criterion="distance")
print(f"{len(set(labels))} clusters at t={t}")


In [0]:
# Custer Graphing
coords = TSNE(n_components=2, metric="cosine", random_state=42).fit_transform(X_centered)

plt.figure(figsize=(10, 7))
for c in sorted(set(labels)):
    mask = labels == c
    plt.scatter(coords[mask, 0], coords[mask, 1], label=f"Cluster {c}", s=50)

plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title(f"{len(set(labels))} clusters at t={t}")
plt.tight_layout()
plt.show()

In [0]:
# Organizing UPC assignment per cluster
clusters_df = spark.createDataFrame(
    [(u, int(l)) for u, l in zip(upcs, labels)],
    ["upc", "cluster"]
)

promoted_upc_clusters_df = promoted_upc_list_vector_df.join(clusters_df, "upc")
promoted_upc_clusters_df.select("upc", "cluster").orderBy("cluster", "upc").display()

# Clustering with UPCs
cluster_assignment_df = clusters_df.groupBy("cluster").agg(
  f.count("upc").alias("n_upcs"), f.collect_list("upc").alias("upcs")).orderBy("cluster")
cluster_assignment_df.display()

In [0]:
# Join UPC Clusters back with ACDS products to see what UPCs fell into each cluster
promoted_upc_clusters_info_df = promoted_upc_clusters_df.join(
    promoted_products_relevant_info,
    promoted_upc_clusters_df["upc"] == promoted_products_relevant_info["gtin_no"],
    "left"
)

promoted_upc_clusters_info_df.display()

In [0]:
cluster_ids = [
    r["cluster"] for r in
    promoted_upc_clusters_info_df.select("cluster").distinct().orderBy("cluster").collect()
]

# Saving every UPC into a dict
cluster_dfs = {
    c: promoted_upc_clusters_info_df.filter(f.col("cluster") == c)
    for c in cluster_ids
}

for c, df in cluster_dfs.items():
    print(c, df.count())

### Initialize Updated KroBERT Parquet with ALL UPCs

In [0]:
# NEW updated Krobert Path w/ latest embedding rundate
BASE_AZURE_DIR = "abfss://uc-utility@sa8451relsciprd.dfs.core.windows.net/relevancy_sciences_volume/foundational_sciences/e451_krobert2vec_pqt"

EMBEDDING_RUNDATE = max(
    f.name.rstrip("/") for f in dbutils.fs.ls(BASE_AZURE_DIR)
    if re.fullmatch(r"\d{8}/?", f.name)
)

FILEPATH = os.path.join(BASE_AZURE_DIR, EMBEDDING_RUNDATE, "krobert.parquet")
krobert_vec_df = (
    spark.read.parquet(FILEPATH)
    .withColumnRenamed("vector", "vector_kro")
    .withColumnRenamed("index", "BAS_CON_UPC_NO")
)
print(f"Row count: {krobert_vec_df.count()}")

krobert_vec_df.display()

In [0]:
dtype = krobert_vec_df.schema["vector_kro"].dataType
if not isinstance(dtype, ArrayType):
    krobert_vec_df = krobert_vec_df.withColumn("vector_kro", vector_to_array("vector_kro"))

In [0]:
# Normalize kroBERT vectors so we can do cosine similarity with promoted upcs
krobert_vec_df = krobert_vec_df.withColumn(
    "norm",
    f.sqrt(f.aggregate("vector_kro", f.lit(0.0), lambda acc, x: acc + x * x))
)

krobert_vec_norm_df = (
    krobert_vec_df
    .withColumn("vector_kro_norm", f.transform("vector_kro", lambda x: x / f.col("norm")))
    .drop("norm")
)

krobert_vec_norm_df.select("BAS_CON_UPC_NO", "vector_kro_norm").display()
print(krobert_vec_norm_df.count())

### Representative Embeddings for Promoted UPCs + Cosine Similarity

In [0]:
print(cluster_dfs.items())

In [0]:
# Create representative embedding here - this time we use centroid of clusters.
mean_embedding_list = []
cluster_ids = []

# Normalize the mean embedding (centroid) here as well, so we can cossim with normalized kroBERT
for c, df in cluster_dfs.items():
    emb = np.array([r["vector"] for r in df.select("vector").collect()])
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

    centroid = emb.mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)  

    mean_embedding_list.append(centroid)
    cluster_ids.append(c)

centroids = np.vstack(mean_embedding_list)   
# shape should be (17, 768)
print(centroids.shape)

In [0]:
# Make as a DF for visibility
schema = StructType([
    StructField("centroid", IntegerType(), False),   # change to StringType if your cluster ids are strings
    StructField("embedding", ArrayType(FloatType()), False),
])

centroid_sdf = spark.createDataFrame(
    [(int(c), v.astype(float).tolist()) for c, v in zip(cluster_ids, centroids)],
    schema=schema,
)

centroid_sdf.display()

### Cosine Similarity Calculation
- Calculating Cossim using normalized vectors

In [0]:
promoted_upcs_sdf = (
    cluster_assignment_df
    .select(f.explode("upcs").alias("BAS_CON_UPC_NO"))
    .distinct()
)

# Cross join every UPC with the 17 centroids and dot product (Cossim)
scores_sdf = (
    krobert_vec_norm_df
    .select("BAS_CON_UPC_NO", f.col("vector_kro_norm").alias("upc_emb"))
    .crossJoin(
        f.broadcast(
            centroid_sdf.select(
                f.col("centroid"),
                f.col("embedding").alias("centroid_emb"),
            )
        )
    )
    .withColumn(
        "cos_sim",
        f.aggregate(
            f.zip_with("upc_emb", "centroid_emb", lambda a, b: a * b),
            f.lit(0.0),
            lambda acc, x: acc + x,
        ),
    )
    .select("BAS_CON_UPC_NO", "centroid", "cos_sim")
)

scores_sdf.display()

In [0]:
# Drop already-promoted UPCs (otherwise cossim will rank those highest)
w = Window.partitionBy("centroid").orderBy(f.col("cos_sim").desc())

# I did where cossim is .6 or greater per cluster (same as prev. logic)
top_per_cluster = (
    scores_sdf
    .join(promoted_upcs_sdf, on="BAS_CON_UPC_NO", how="left_anti")
    .withColumn("rank", f.row_number().over(w))
    .where(f.col("cos_sim") >= 0.60)          
    .join(
        cluster_assignment_df.select(
            f.col("cluster").alias("centroid"),
            f.col("n_upcs").alias("cluster_size"),
        ),
        on="centroid",
        how="left",
    )
    .orderBy("centroid", "rank")
)

top_per_cluster.display()

In [0]:
cossim_upcs_per_cluster = top_per_cluster.select("centroid", "BAS_CON_UPC_NO", "cos_sim", "rank")
cossim_upcs_per_cluster.display()

### Bring in YNB and Blend Cossim/YNB/Relevant UPC Count

In [0]:
# Initialize Most Recent Fiscal Week for YNB
fiscal_week = spark.table("personalization_prd.relevancy_sciences.pzn_ynb_fiscal_week").agg(f.max("FIS_WEEK_ID")).first()[0]

In [0]:
# YNB Df
ynb_df = spark.table("personalization_prd.relevancy_sciences.pzn_ynb_prd_final_rank").filter(f"fiscal_week = {fiscal_week}").select("EHHN", "BAS_CON_UPC_NO", "fiscal_week", "PRD_SCORE", "PRD_RANK")
ynb_df.display()

In [0]:
# Normalize YNB score for Blending
lo, hi = ynb_df.approxQuantile("PRD_SCORE", [0.01, 0.99], 0.001)

ynb_norm_df = ynb_df.withColumn(
    "prd_norm",
    f.least(
        f.lit(1.0),
        f.greatest(f.lit(0.0), (f.col("PRD_SCORE") - f.lit(lo)) / f.lit(hi - lo)),
    ),
)

ynb_norm_df.display()

In [0]:
# Normalize Cosine Similarity Score for Blending
w_c = Window.partitionBy("centroid")

cossim_norm_df = (
    cossim_upcs_per_cluster
    .withColumn("cs_min", f.min("cos_sim").over(w_c))
    .withColumn("cs_max", f.max("cos_sim").over(w_c))
    .withColumn(
        "cos_norm",
        f.when(
            f.col("cs_max") > f.col("cs_min"),
            (f.col("cos_sim") - f.col("cs_min")) / (f.col("cs_max") - f.col("cs_min")),
        ).otherwise(f.lit(1.0)),
    )
    .drop("cs_min", "cs_max")
)

cossim_norm_df.display()

In [0]:
# Pull Households who purchased relevant cossim scored UPCs
# 40% Cosine Similiarity, 40% YNB score, 20% Number of Qualifying UPCs per HH

W_COS, W_YNB, W_CNT = 0.4, 0.4, 0.2  

# 1) UPC-level score: cosine + YNB only (start from ynb_norm_df and cossim_norm_df)
hh_upc = (
    ynb_norm_df
    .join(f.broadcast(cossim_norm_df), on="BAS_CON_UPC_NO", how="inner")
    .withColumn(
        "upc_score",
        f.lit(W_COS) * f.col("cos_norm") + f.lit(W_YNB) * f.col("prd_norm"),
    )
)

In [0]:
# Group by centroid / similar upc : collapse UPC-level blended scores
hh_blended = (
    hh_upc
    .groupBy("centroid", "EHHN")
    .agg(
        f.max("upc_score").alias("ynb_cossim_blended_score"),                
        f.max_by("BAS_CON_UPC_NO", "upc_score").alias("top_upc"),
        f.countDistinct("BAS_CON_UPC_NO").alias("n_qualifying_upcs"),
    )
)

In [0]:
# Blend YNB, Cosine Similarity, and Number of relevant UPCs per HH
window_centroid = Window.partitionBy("centroid")

hh_blended_cossim_ynb_upccount = (
    hh_blended
    .withColumn(
        "number_hh_upcs",
        # standardize by dividing each household’s count by the largest count in its centroid,
        f.col("n_qualifying_upcs") / f.max("n_qualifying_upcs").over(window_centroid),
    )
    .withColumn(
        "ynb_cossim_upc_final_blended_score",
        f.col("ynb_cossim_blended_score") + f.lit(W_CNT) * f.col("number_hh_upcs"),
    )
)

In [0]:
# Rank households within each centroid
window_hh = Window.partitionBy("centroid").orderBy(
    f.col("ynb_cossim_upc_final_blended_score").desc(), f.col("EHHN")
)
hh_ranked = hh_blended_cossim_ynb_upccount.withColumn("hh_rank", f.row_number().over(window_hh))

hh_ranked_final = hh_ranked.select(
  "centroid", "EHHN", "number_hh_upcs", "ynb_cossim_blended_score", "ynb_cossim_upc_final_blended_score", "hh_rank")

In [0]:
hh_ranked_final.where(f.col("centroid") == 3).orderBy("hh_rank").display()

### Best/Better/Good Decile Calculation

In [0]:
# Good Better Best Decile Calculation
window_decile = Window.partitionBy("centroid").orderBy(
    f.col("ynb_cossim_upc_final_blended_score").desc(), f.col("EHHN")
)

hh_deciles = (
    hh_ranked_final
    .withColumn("decile_raw", f.ntile(10).over(window_decile))   
    .withColumn("HH_Decile", 11 - f.col("decile_raw"))    
    .withColumn(
        "HH_Bucket",
        f.when(f.col("HH_Decile") == 10, "Best")
         .when(f.col("HH_Decile").between(6, 9), "Better")
         .otherwise("Good"),                                      
    )
    .drop("decile_raw")
)

# View one centroid
hh_deciles.where(f.col("centroid") == 3).orderBy("hh_rank").limit(1000).display()

In [0]:
# Extract only "BEST" HHs
best_hhs = hh_deciles.where(f.col("HH_Bucket") == "Best").orderBy("centroid", "hh_rank")
print(f"Total 'Best' HHs: {best_hhs.count():,}")

best_hhs.groupBy("centroid").count().orderBy("centroid").display()
best_hhs.limit(1000).display()